# Data Understanding — Diabetes 130-US Hospitals

Notebook này chỉ dùng để khám phá và kiểm tra dữ liệu.

Production logic nằm trong:

- `src/data/ingestion.py`
- `src/data/validation.py`
- `src/data/splitting.py`
- `src/features/build_features.py`
- `src/features/preprocessing.py`

Notebook không chứa logic training production.

In [ ]:
// Xác định project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Không tìm thấy thư mục src/. Hãy mở notebook từ project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
// Import module production
from src.data.ingestion import load_raw_dataset
from src.data.validation import (
    load_quality_config,
    validate_dataframe,
)
from src.features.build_features import build_features

In [ ]:
Đọc raw dataset
RAW_DATA_PATH = PROJECT_ROOT / "data/raw/diabetic_data.csv"

df = load_raw_dataset(RAW_DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

In [ ]:
//Chạy validation
QUALITY_CONFIG_PATH = PROJECT_ROOT / "configs/data_quality.yaml"

quality_config = load_quality_config(QUALITY_CONFIG_PATH)
validation_report = validate_dataframe(df, quality_config)

validation_report

print("Schema passed:", validation_report["schema_passed"])
print("Errors:", validation_report["errors"])
print("Warnings:", validation_report["warnings"])
print("Duplicate count:", validation_report["duplicate_count"])
print("Positive rate:", validation_report["label_positive_rate"])

In [ ]:
//EDA cơ bản
schema_summary = df.dtypes.astype(str).to_frame("dtype")
schema_summary["missing_count"] = df.isna().sum()
schema_summary["missing_rate"] = df.isna().mean()
schema_summary["unique_count"] = df.nunique(dropna=False)

schema_summary.sort_values("missing_rate", ascending=False).head(30)

df["readmitted"].value_counts(dropna=False)
df["readmitted"].value_counts(normalize=True, dropna=False)
df.duplicated().sum()

In [ ]:
// Feature engineering
feature_df = build_features(df)

new_columns = sorted(set(feature_df.columns) - set(df.columns))

print("New engineered features:")
new_columns

Ghi rõ phần không thuộc notebook này
## Out of scope

Các phần sau thuộc pipeline hoặc role khác:

- Train/validation/test split: `src/data/splitting.py`
- Fit preprocessing: `src/features/preprocessing.py`
- Dummy Classifier: Member B
- Logistic Regression: Member B
- Random Forest/XGBoost: Member B
- Model evaluation chính thức: Member B

Notebook này không được dùng làm nguồn production duy nhất.